# Task 1: Exploratory Data Analysis and Preprocessing

**Objective**: Understand the CFPB complaint dataset, clean it, and prepare it for the RAG pipeline.

**Product Categories of Interest** (as per CrediTrust):
1. Credit Cards
2. Personal Loans
3. Buy Now Pay Later
4. Savings Accounts
5. Money Transfers

I will filter the dataset to these five categories and clean the complaint narratives.

# 1. Setting up the environment

In [ ]:
print("Setting Up and importing...")
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
%matplotlib inline

print("Libraries imported successfully!")

## 2. Load the Dataset Efficiently (Using Chunking)

The dataset is large. To avoid memory issues, we will load it in chunks using `pandas.read_csv` with `chunksize`. We'll process each chunk to:
- Collect product distribution counts.
- Collect narrative length statistics.
- Filter and keep only rows matching our five product categories and having a narrative.

Finally, we'll combine the filtered rows into a single DataFrame and save it.

In [ ]:
# Mapping from dataset product names to our standardized categories
product_mapping = {
    'Credit card': 'Credit card',
    'Credit card or prepaid card': 'Credit card',
    'Prepaid card': 'Credit card',  # Prepaid cards are often considered credit products
    'Personal loan': 'Personal loan',
    'Payday loan, title loan, or personal loan': 'Personal loan',
    'Payday loan, title loan, personal loan, or advance loan': 'Personal loan',
    'Payday loan': 'Personal loan',  # though payday is different, we'll map to personal loan for this exercise
    'Buy Now Pay Later': 'Buy Now Pay Later',  # may not exist
    'Savings account': 'Savings account',
    'Checking or savings account': 'Savings account',  # includes both, but we'll count as savings for now
    'Money transfer': 'Money transfer',
    'Money transfers': 'Money transfer',
    'Money transfer, virtual currency, or money service': 'Money transfer'
}

# Our target categories
target_categories = list(product_mapping.values())  # ['Credit card', 'Personal loan', 'Buy Now Pay Later', 'Savings account', 'Money transfer']

In [ ]:
# Define file path
file_path = '../data/raw/complaints.csv'  

# Define target products
target_products = [
    'Credit card',
    'Personal loan',
    'Buy Now Pay Later',
    'Savings account',
    'Money transfer'
]

# Initialize empty list to store filtered chunks
filtered_chunks = []

# Initialize counters for overall statistics
total_rows = 0
product_counter = {}
mapped_product_counter ={} # for mapped counts
narrative_word_counts = []

chunk_size = 100000
for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size, low_memory=False)):
    print(f"Processing chunk {i+1}...")
    total_rows += len(chunk)
    
    # Original product counts
    orig_counts = chunk['Product'].value_counts().to_dict()
    for prod, cnt in orig_counts.items():
        product_counter[prod] = product_counter.get(prod, 0) + cnt
    
    # Map products to our categories
    chunk['mapped_product'] = chunk['Product'].map(product_mapping)
    
    # Filter rows that have a mapped product (not NaN) and have narrative
    mask = (chunk['mapped_product'].notna()) & (chunk['Consumer complaint narrative'].notna())
    filtered = chunk[mask].copy()
    
    if not filtered.empty:
        # Update mapped product counts
        mapped_counts = filtered['mapped_product'].value_counts().to_dict()
        for prod, cnt in mapped_counts.items():
            mapped_product_counter[prod] = mapped_product_counter.get(prod, 0) + cnt
        
        # Word counts
        word_counts = filtered['Consumer complaint narrative'].apply(lambda x: len(str(x).split()))
        narrative_word_counts.extend(word_counts.tolist())
        
        filtered_chunks.append(filtered)
    
    # Optional early break for testing
    # if i > 5: break

In [ ]:
if filtered_chunks:
    filtered_df = pd.concat(filtered_chunks, ignore_index=True)
    print(f"Total filtered rows: {len(filtered_df)}")
    print("\nDistribution by mapped product:")
    print(filtered_df['mapped_product'].value_counts())
    print("\nDistribution by original product (within filtered set):")
    print(filtered_df['Product'].value_counts())
else:
    filtered_df = pd.DataFrame()
    print("No matching rows found.")

### Product Distribution Summary

In [ ]:
# Convert product_counter to a Series for easy viewing
product_series = pd.Series(product_counter).sort_values(ascending=False)
print("Product distribution across entire dataset (top 20):")
print(product_series.head(20))

### Narrative Length Statistics (from filtered rows only)
We need to understand the length distribution of complaint narratives to guide chunking later.

In [ ]:
if narrative_word_counts:
    word_counts_series = pd.Series(narrative_word_counts)
    print(word_counts_series.describe())
    
    # Plot histogram (capped at 500 for readability)
    plt.figure(figsize=(10,5))
    plt.hist(word_counts_series.clip(upper=500), bins=50, edgecolor='black')
    plt.title('Distribution of Complaint Narrative Word Count (capped at 500)')
    plt.xlabel('Word Count')
    plt.ylabel('Frequency')
    plt.show()
else:
    print("No filtered rows found with narratives.")

In [ ]:
# Check extremes: very short (<5 words) and very long (>1000 words)
short_narratives = filtered_df[filtered_df['narrative_word_counts'] < 5].shape[0]
long_narratives = filtered_df[filtered_df['narrative_word_counts'] > 1000].shape[0]
total_with_narrative = filtered_df['Consumer complaint narrative'].notna().sum()
 
print(f"Total complaints with narrative: {total_with_narrative}")
print(f"Very short narratives (<5 words): {short_narratives} ({short_narratives/total_with_narrative*100:.2f}%)")
print(f"Very long narratives (>1000 words): {long_narratives} ({long_narratives/total_with_narrative*100:.2f}%)")

## 3. Combine Filtered Chunks and Save

In [ ]:
# Concatenate all filtered chunks
if filtered_chunks:
    df = pd.concat(filtered_chunks, ignore_index=True)
    print(f"Total filtered rows: {len(filtered_df)}")
    print(f"Product distribution in filtered set:")
    print(filtered_df['Product'].value_counts())
else:
    print("No rows matched the filter criteria.")
    filtered_df = pd.DataFrame()  # empty dataframe

## 4. Initial Data inspection

In [ ]:
# Display first few rows to begin with
filtered_df.head()

In [ ]:
# check column names and data types
filtered_df.info()

In [ ]:
# Check for missing values (as percentage)
missing = filtered_df.isnull().sum() / len(filtered_df) * 100
missing[missing > 0].sort_values(ascending=False)

**Observations**:
- The dataset has many columns; we are primarily interested in:
  - `Product` (or similar) – to filter by our five categories.
  - `Consumer complaint narrative` – the text we will use.
  - Other metadata like `Issue`, `Company`, `State`, `Date received` for later use.
- We see that a significant portion of rows lack a complaint narrative. These will be removed.

## 4. Distribution accross products

In [ ]:
# Count complaints per product
product_counts = filtered_df['Product'].value_counts()
print("Product distribution:")
print(product_counts)

# Plot top 20 products
plt.figure(figsize=(12,6))
product_counts.head(20).plot(kind='bar')
plt.title('Top 20 Product Categories by Number of Complaints')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Define our five product categories (as they might appear in the dataset)
target_products = [
    'Credit card',
    'Personal loan',
    'Buy Now Pay Later',   # May not exist in older CFPB data
    'Savings account',
    'Money transfer'
]

# Check which of these are present in the actual data
present_products = [p for p in target_products if p in product_counts.index]
missing_products = [p for p in target_products if p not in product_counts.index]

print(f"Present in dataset: {present_products}")
print(f"Not found: {missing_products}")

**Note**: "Buy Now Pay Later" is a newer product and may not appear in the historical CFPB data. For the purposes of this project, we will still include it in our filtering step, but it will result in zero records. This is acceptable as the model should be prepared to handle such cases in the future.

## 6. Missing Narratives

We need to know how many complaints lack a narrative, as these will be dropped.

In [ ]:
missing_narrative = df['Consumer complaint narrative'].isna().sum()
total_rows = len(df)

print(f"Rows missing narrative: {missing_narrative} ({missing_narrative/total_rows*100:.2f}%)")
print(f"Rows with narrative: {total_rows - missing_narrative} ({(total_rows - missing_narrative)/total_rows*100:.2f}%)")

## 7. Filtering to Target Products and Non-Empty Narratives

We will:
- Keep only rows where `Product` is one of our five categories.
- Keep only rows where `Consumer complaint narrative` is not empty.

In [ ]:
# Filter by product (using the present_products list, but we can also include all target_products; missing ones will just yield no rows)
filtered_df = df[df['Product'].isin(target_products)].copy()

# Remove rows with missing narrative
filtered_df = filtered_df[filtered_df['Consumer complaint narrative'].notna()].copy()

print(f"After filtering: {filtered_df.shape[0]} rows remain.")
print(f"Product distribution in filtered set:")
print(filtered_df['Product'].value_counts())

## 8. Text Cleaning

We'll clean the complaint narratives to improve embedding quality.  
Typical steps:
- Convert to lowercase
- Remove extra whitespace
- Remove special characters/punctuation (optional, can keep sentence structure)
- Remove common boilerplate phrases (e.g., "I am writing to file a complaint...") – this may be dataset-specific.

We'll implement a basic cleaning function.

In [ ]:
def clean_text(text):
    """
    Basic text cleaning:
    - Lowercase
    - Remove special characters (keep letters, numbers, spaces)
    - Remove extra whitespace
    """
    if not isinstance(text, str):
        return ""
    # Lowercase
    text = text.lower()
    # Remove special characters (keep alphanumeric and spaces)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
filtered_df['cleaned_narrative'] = filtered_df['Consumer complaint narrative'].apply(clean_text)

# Show example before/after
sample_idx = filtered_df.sample(1, random_state=42).index[0]
print("ORIGINAL:")
print(filtered_df.loc[sample_idx, 'Consumer complaint narrative'][:500])
print("\nCLEANED:")
print(filtered_df.loc[sample_idx, 'cleaned_narrative'][:500])

**Optional**: You may add removal of common boilerplate phrases if they appear frequently. For now, this basic cleaning is sufficient.

## 9. Save Cleaned Dataset

We will save the filtered and cleaned dataframe to `data/processed/filtered_complaints.csv`.  
We'll keep all original columns plus the cleaned narrative.

In [ ]:
# Save to processed folder
output_path = '../data/processed/filtered_complaints.csv'
filtered_df.to_csv(output_path, index=False)
print(f"Cleaned data saved to {output_path}")

## 10. Summary of Key Findings

*[Write 2-3 paragraphs here summarizing your observations from the EDA. Include things like:]*

- Total complaints in original dataset, number with narratives.
- How many complaints per product category, and how filtering affected the size.
- Distribution of narrative lengths: typical range, presence of very short/long narratives.
- Any data quality issues encountered.